<a href="https://colab.research.google.com/github/Vaquinhas/Service-Ticket-Data-Pipeline/blob/main/prj/Project-1/project_1_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Applied Programming for AI

## Mini-project 1 — Service Ticket Data Pipeline

### Learning outcomes
- Turn data quality rules into a reusable NumPy/Pandas workflow.
- Preserve rejected data, transformation flags and an audit trail.
- Check edge cases and export reproducible results.

### Summary:

1. Diagnosis
2. Rejection Policy
3. Category Normalization
4. Numeric Treatment
5. Features
6. Reusable Function
7. Edge Cases
8. Export and Report

---

Read [BRIEF.md](BRIEF.md) before coding. Keep the synthetic input unchanged. Complete each task and restart/run from the beginning before submitting. **Contribution: 1.0/20 (5%).**

### Importing libraries and loading the data

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

candidates=[Path('service_tickets.csv'),Path('prj/Project-1/service_tickets.csv')]
source=next((p for p in candidates if p.is_file()),None)
if source is None: raise FileNotFoundError('Open from the project folder or upload service_tickets.csv to Colab')
raw=pd.read_csv(source)
print(raw.head())

  ticket_id   opened_on channel priority handling_minutes  satisfaction
0      T001  2026-10-01   email   medium             42.2           2.0
1      T002  2026-10-01   phone     high             76.3           3.0
2      T003  2026-10-01     web      low              NaN           4.0
3      T004  2026-10-01   email   medium            118.0           5.0
4      T005  2026-10-02   phone     high             99.5           1.0


---

## 1. Diagnosis

Report shape, schema, missingness, exact duplicates and repeated IDs. Explain the observation unit.

In [2]:
# 1. Diagnosis: inspect raw dataset without mutating it
print("Shape (rows, columns):", raw.shape)
print("\nSchema (column data types):", raw.dtypes)

print("\nMissing values per column:")
print(raw.isna().sum())

# Count extra exact duplicate rows
exact_dup_count = raw.duplicated().sum()
print(f"\nExact duplicate rows (extra copies): {exact_dup_count}")

# Inspect repeated ticket_id values (excluding missing IDs)
non_null_ids = raw["ticket_id"].notna()
repeated_id_rows = raw[non_null_ids & raw.duplicated(subset=["ticket_id"], keep=False)]
print(f"Rows with repeated ticket_id (in raw): {len(repeated_id_rows)}")

# Inspect repeated ticket_id values after removing exact duplicates (conflicting IDs)
deduped_preview = raw.drop_duplicates()
conflicting_preview = deduped_preview[
    deduped_preview["ticket_id"].notna()
    & deduped_preview.duplicated(subset=["ticket_id"], keep=False)
]
print(f"Rows with conflicting ticket_id (after removing exact copies): {len(conflicting_preview)}")

Shape (rows, columns): (84, 6)

Schema (column data types): ticket_id            object
opened_on            object
channel              object
priority             object
handling_minutes     object
satisfaction        float64
dtype: object

Missing values per column:
ticket_id           1
opened_on           0
channel             1
priority            1
handling_minutes    1
satisfaction        1
dtype: int64

Exact duplicate rows (extra copies): 3
Rows with repeated ticket_id (in raw): 8
Rows with conflicting ticket_id (after removing exact copies): 2


---

## 2. Rejection policy

Remove exact copies. Identify conflicting IDs and invalid IDs/dates. Preserve rejected source rows and all reasons.

In [3]:
# work on a copy so raw is never mutated
working_df = raw.copy()

# remove exact duplicate rows first
deduped_df = working_df.drop_duplicates().copy()
exact_duplicates_removed = len(working_df) - len(deduped_df)

# validate ticket_id (missing, malformed, or conflicting)
id_as_str = deduped_df["ticket_id"].astype("string")
missing_id_mask = deduped_df["ticket_id"].isna() | (id_as_str.str.strip() == "")
malformed_id_mask = (~missing_id_mask) & (~id_as_str.str.fullmatch(r"T\d{3}").fillna(False))
conflicting_id_mask = (~missing_id_mask) & deduped_df.duplicated(subset=["ticket_id"], keep=False)

# validate opened_on (missing, unparseable, or outside October 2026)
date_as_str = deduped_df["opened_on"].astype("string")
missing_date_mask = deduped_df["opened_on"].isna() | (date_as_str.str.strip() == "")
parsed_dates = pd.to_datetime(deduped_df["opened_on"], errors="coerce")
unparseable_date_mask = (~missing_date_mask) & parsed_dates.isna()
out_of_october_mask = parsed_dates.notna() & ~(
    (parsed_dates.dt.year == 2026) & (parsed_dates.dt.month == 10)
)

# combine all rejection reasons into a single Boolean mask
reason_flags = pd.DataFrame(
    {
        "missing_id": missing_id_mask,
        "malformed_id": malformed_id_mask,
        "conflicting_id": conflicting_id_mask,
        "missing_date": missing_date_mask,
        "unparseable_date": unparseable_date_mask,
        "out_of_october_date": out_of_october_mask,
    },
    index=deduped_df.index,
)
reject_mask = reason_flags.any(axis=1)

# Preserve original rejected rows and record every applicable reason
rejected_df = deduped_df.loc[reject_mask].copy()
if reject_mask.any():
    rejected_df["rejection_reason"] = reason_flags.loc[reject_mask].apply(
        lambda row: "; ".join(row.index[row]), axis=1
    )
else:
    rejected_df["rejection_reason"] = pd.Series(dtype="object")

# Keep retained rows and assign parsed dates
clean_df = deduped_df.loc[~reject_mask].copy()
clean_df["opened_on"] = parsed_dates.loc[~reject_mask]

# Reset indices for clean tables
rejected_df = rejected_df.reset_index(drop=True)
clean_df = clean_df.reset_index(drop=True)

print(f"Raw: {len(raw)} | Exact dupes removed: {exact_duplicates_removed} | "
      f"Rejected: {len(rejected_df)} | Retained: {len(clean_df)}")
print("\nRejected sample:")
print(rejected_df[["ticket_id", "opened_on", "rejection_reason"]].head())

Raw: 84 | Exact dupes removed: 3 | Rejected: 4 | Retained: 77

Rejected sample:
  ticket_id   opened_on  rejection_reason
0      T034  2026-10-32  unparseable_date
1       NaN  2026-10-09        missing_id
2      T040  2026-10-10    conflicting_id
3      T040  2026-10-10    conflicting_id


---

## 3. Category normalization

Normalize channel and priority and preserve unknown flags according to the brief.

In [4]:
# 3. Category normalization
valid_channels = {"email", "phone", "web"}
valid_priorities = {"low", "medium", "high"}

norm_channel = clean_df["channel"].astype("string").str.strip().str.lower()
is_valid_channel = norm_channel.isin(valid_channels).fillna(False)
clean_df["channel"] = norm_channel.where(is_valid_channel, "unknown").astype(str)
clean_df["channel_unknown"] = (~is_valid_channel).astype(bool)

norm_priority = clean_df["priority"].astype("string").str.strip().str.lower()
is_valid_priority = norm_priority.isin(valid_priorities).fillna(False)
clean_df["priority"] = norm_priority.where(is_valid_priority, "unknown").astype(str)
clean_df["priority_unknown"] = (~is_valid_priority).astype(bool)

print("Channel counts:\n", clean_df["channel"].value_counts(dropna=False))


Channel counts:
 channel
email      26
phone      25
web        24
unknown     2
Name: count, dtype: int64


---

## 4. Numeric treatment

Treat durations and satisfaction differently. Preserve flags and explain why ratings must not be imputed.

In [5]:
# durations (impute median) vs satisfaction (do NOT impute)
# handling_minutes: nonnegative finite numbers (including 0 and decimals) are valid
handling_num = pd.to_numeric(clean_df["handling_minutes"], errors="coerce").astype(float)
valid_handling_mask = np.isfinite(handling_num) & (handling_num >= 0)

# Set invalid/negative/non-finite values to NaN and flag rows requiring imputation
handling_clean = handling_num.where(valid_handling_mask, np.nan)
clean_df["handling_imputed"] = (~valid_handling_mask).astype(bool)

# Compute median strictly from valid durations in retained rows
valid_durations = handling_clean.dropna()
if valid_durations.empty:
    raise ValueError(
        "No valid handling_minutes remain in retained rows; cannot compute imputation median."
    )

duration_median = float(valid_durations.median())
clean_df["handling_minutes"] = handling_clean.fillna(duration_median)

# satisfaction: only integers from 1 to 5 are valid; do NOT impute
sat_num = pd.to_numeric(clean_df["satisfaction"], errors="coerce").astype(float)
sat_finite = sat_num.where(np.isfinite(sat_num), np.nan)
valid_sat_mask = (
    sat_finite.notna()
    & (sat_finite >= 1)
    & (sat_finite <= 5)
    & (sat_finite == np.floor(sat_finite))
)

# Store as nullable integer (Int64) so valid ratings stay integers and missing stay <NA>
clean_df["satisfaction"] = sat_finite.where(valid_sat_mask, np.nan).astype("Int64")
clean_df["satisfaction_unavailable"] = (~valid_sat_mask).astype(bool)

print(f"Duration median used for imputation: {duration_median}")
print(f"Imputed durations: {clean_df['handling_imputed'].sum()}")
print(f"Unavailable satisfaction ratings: {clean_df['satisfaction_unavailable'].sum()}")

Duration median used for imputation: 59.0
Imputed durations: 3
Unavailable satisfaction ratings: 3


In [6]:
print(clean_df["channel"].value_counts(dropna=False))
print("\nUnexpected channel values:")
print(clean_df.loc[~clean_df["channel"].isin(["email", "phone", "web", "unknown"]), ["ticket_id", "channel"]])

channel
email      26
phone      25
web        24
unknown     2
Name: count, dtype: int64

Unexpected channel values:
Empty DataFrame
Columns: [ticket_id, channel]
Index: []


---

## 5. Features

Compute safe duration scaling and the four required channel indicators; verify identifier alignment.

In [7]:
# safe z-score scaling and four channel indicators
duration_mean = float(clean_df["handling_minutes"].mean())
duration_std = float(clean_df["handling_minutes"].std(ddof=0))

# Use safe denominator of 1 when the duration column is constant
if clean_df["handling_minutes"].nunique() <= 1 or np.isclose(duration_std, 0.0):
    clean_df["handling_zscore"] = 0.0
else:
    clean_df["handling_zscore"] = (clean_df["handling_minutes"] - duration_mean) / duration_std

# Build separate feature table (exclude satisfaction by design)
features_df = pd.DataFrame(
    {
        "ticket_id": clean_df["ticket_id"].values,
        "handling_zscore": clean_df["handling_zscore"].values,
        "channel_email": (clean_df["channel"] == "email").astype(int).values,
        "channel_phone": (clean_df["channel"] == "phone").astype(int).values,
        "channel_web": (clean_df["channel"] == "web").astype(int).values,
        "channel_unknown": (clean_df["channel"] == "unknown").astype(int).values,
    }
)

# Verify identifier alignment and one-hot exclusivity
indicator_cols = ["channel_email", "channel_phone", "channel_web", "channel_unknown"]
assert (features_df["ticket_id"].values == clean_df["ticket_id"].values).all(), "ID mismatch!"
assert (features_df[indicator_cols].sum(axis=1) == 1).all(), "Each row must have exactly 1 active channel!"
assert "satisfaction" not in features_df.columns, "Satisfaction must not be in features!"

print("Features verification passed.")
print(features_df.head())

Features verification passed.
  ticket_id  handling_zscore  channel_email  channel_phone  channel_web  \
0      T001        -0.568376              1              0            0   
1      T002         0.546176              0              1            0   
2      T003        -0.019271              0              0            1   
3      T004         1.909131              1              0            0   
4      T005         1.304463              0              1            0   

   channel_unknown  
0                0  
1                0  
2                0  
3                0  
4                0  


---

## 6. Reusable function

Consolidate your work into clean_tickets(raw), returning clean, rejected, features and audit. Do not mutate the input.

In [8]:
# 6. Reusable pipeline function
def clean_tickets(raw_df: pd.DataFrame):
    """Clean service ticket records and return clean, rejected, features tables and audit dict."""
    # Never mutate the input DataFrame
    working_df = raw_df.copy()
    raw_count = int(len(working_df))

    # 1. Remove exact duplicate rows
    deduped_df = working_df.drop_duplicates().copy()
    exact_duplicates_removed = int(raw_count - len(deduped_df))

    # 2. Identify rejection conditions
    id_as_str = deduped_df["ticket_id"].astype("string")
    missing_id_mask = deduped_df["ticket_id"].isna() | (id_as_str.str.strip() == "")
    malformed_id_mask = (~missing_id_mask) & (~id_as_str.str.fullmatch(r"T\d{3}").fillna(False))
    conflicting_id_mask = (~missing_id_mask) & deduped_df.duplicated(subset=["ticket_id"], keep=False)

    date_as_str = deduped_df["opened_on"].astype("string")
    missing_date_mask = deduped_df["opened_on"].isna() | (date_as_str.str.strip() == "")
    parsed_dates = pd.to_datetime(deduped_df["opened_on"], errors="coerce")
    unparseable_date_mask = (~missing_date_mask) & parsed_dates.isna()
    out_of_october_mask = parsed_dates.notna() & ~(
        (parsed_dates.dt.year == 2026) & (parsed_dates.dt.month == 10)
    )

    reason_flags = pd.DataFrame(
        {
            "missing_id": missing_id_mask,
            "malformed_id": malformed_id_mask,
            "conflicting_id": conflicting_id_mask,
            "missing_date": missing_date_mask,
            "unparseable_date": unparseable_date_mask,
            "out_of_october_date": out_of_october_mask,
        },
        index=deduped_df.index,
    )
    reject_mask = reason_flags.any(axis=1)

    # Preserve rejected raw rows and all applicable reasons
    rejected = deduped_df.loc[reject_mask].copy()
    if reject_mask.any():
        rejected["rejection_reason"] = reason_flags.loc[reject_mask].apply(
            lambda row: "; ".join(row.index[row]), axis=1
        )
    else:
        rejected["rejection_reason"] = pd.Series(dtype="object")
    rejected = rejected.reset_index(drop=True)

    # Retain valid rows
    clean = deduped_df.loc[~reject_mask].copy()
    clean["opened_on"] = parsed_dates.loc[~reject_mask]
    clean = clean.reset_index(drop=True)

    # 3. Normalize categories (channel & priority)
    valid_channels = {"email", "phone", "web"}
    valid_priorities = {"low", "medium", "high"}

    norm_channel = clean["channel"].astype("string").str.strip().str.lower()
    is_valid_channel = norm_channel.isin(valid_channels).fillna(False)
    clean["channel"] = np.where(is_valid_channel, norm_channel, "unknown")
    clean["channel_unknown"] = (~is_valid_channel).astype(bool)

    norm_priority = clean["priority"].astype("string").str.strip().str.lower()
    is_valid_priority = norm_priority.isin(valid_priorities).fillna(False)
    clean["priority"] = np.where(is_valid_priority, norm_priority, "unknown")
    clean["priority_unknown"] = (~is_valid_priority).astype(bool)

    # 4. Numeric treatment: handling_minutes
    handling_num = pd.to_numeric(clean["handling_minutes"], errors="coerce").astype(float)
    valid_handling_mask = np.isfinite(handling_num) & (handling_num >= 0)
    handling_clean = handling_num.where(valid_handling_mask, np.nan)
    clean["handling_imputed"] = (~valid_handling_mask).astype(bool)

    valid_durations = handling_clean.dropna()
    if valid_durations.empty:
        raise ValueError(
            "No valid handling_minutes remain in retained rows; cannot compute imputation median."
        )

    duration_median = float(valid_durations.median())
    clean["handling_minutes"] = handling_clean.fillna(duration_median)

    # 5. Numeric treatment: satisfaction (no imputation)
    sat_num = pd.to_numeric(clean["satisfaction"], errors="coerce").astype(float)
    sat_finite = sat_num.where(np.isfinite(sat_num), np.nan)
    valid_sat_mask = (
        sat_finite.notna()
        & (sat_finite >= 1)
        & (sat_finite <= 5)
        & (sat_finite == np.floor(sat_finite))
    )
    clean["satisfaction"] = sat_finite.where(valid_sat_mask, np.nan).astype("Int64")
    clean["satisfaction_unavailable"] = (~valid_sat_mask).astype(bool)

    # 6. Duration z-score (population std, safe denominator of 1 for constant column)
    duration_mean = float(clean["handling_minutes"].mean())
    duration_std = float(clean["handling_minutes"].std(ddof=0))
    if clean["handling_minutes"].nunique() <= 1 or np.isclose(duration_std, 0.0):
        clean["handling_zscore"] = 0.0
    else:
        clean["handling_zscore"] = (clean["handling_minutes"] - duration_mean) / duration_std

    # 7. Separate feature table
    features = pd.DataFrame(
        {
            "ticket_id": clean["ticket_id"].values,
            "handling_zscore": clean["handling_zscore"].values,
            "channel_email": (clean["channel"] == "email").astype(int).values,
            "channel_phone": (clean["channel"] == "phone").astype(int).values,
            "channel_web": (clean["channel"] == "web").astype(int).values,
            "channel_unknown": (clean["channel"] == "unknown").astype(int).values,
        }
    )

    # 8. Build JSON-serializable audit dictionary
    audit = {
        "raw_count": raw_count,
        "exact_duplicates_removed": exact_duplicates_removed,
        "rejected_count": int(len(rejected)),
        "retained_count": int(len(clean)),
        "channel_unknown_count": int(clean["channel_unknown"].sum()),
        "priority_unknown_count": int(clean["priority_unknown"].sum()),
        "handling_imputed_count": int(clean["handling_imputed"].sum()),
        "satisfaction_unavailable_count": int(clean["satisfaction_unavailable"].sum()),
        "handling_fill_median": duration_median,
        "handling_mean": duration_mean,
        "handling_std_pop": duration_std,
    }

    return clean, rejected, features, audit


# Run pipeline on the full dataset and confirm raw is not mutated
raw_snapshot = raw.copy(deep=True)
clean, rejected, features, audit = clean_tickets(raw)
assert raw.equals(raw_snapshot), "Input DataFrame was mutated!"

print("Audit summary:")
print(json.dumps(audit, indent=2))

Audit summary:
{
  "raw_count": 84,
  "exact_duplicates_removed": 3,
  "rejected_count": 4,
  "retained_count": 77,
  "channel_unknown_count": 2,
  "priority_unknown_count": 2,
  "handling_imputed_count": 3,
  "satisfaction_unavailable_count": 3,
  "handling_fill_median": 59.0,
  "handling_mean": 59.589610389610385,
  "handling_std_pop": 30.595274163617248
}


---

## 7. Edge cases

Test the two-ticket constant-duration example, conflicting IDs and the no-valid-duration error from the brief.

In [9]:
# 7. Edge cases: verify all contract rules from the brief

# Case 1: Two-ticket constant-duration worked example
case_const = pd.DataFrame(
    [
        {"ticket_id": "T001", "opened_on": "2026-10-05", "channel": "email", "priority": "low", "handling_minutes": 10, "satisfaction": 4},
        {"ticket_id": "T002", "opened_on": "2026-10-12", "channel": "web", "priority": "high", "handling_minutes": np.nan, "satisfaction": 5},
    ]
)
c1, r1, f1, a1 = clean_tickets(case_const)
assert len(r1) == 0, "Neither ticket should be rejected"
assert c1["handling_minutes"].tolist() == [10.0, 10.0]
assert c1["handling_imputed"].tolist() == [False, True]
assert c1["handling_zscore"].tolist() == [0.0, 0.0]
assert f1["handling_zscore"].tolist() == [0.0, 0.0]
assert (f1[["channel_email", "channel_phone", "channel_web", "channel_unknown"]].sum(axis=1) == 1).all()

# Case 2: Exact duplicate, conflicting IDs, date errors, multi-reason rejection,
#         zero/fractional durations, and fractional satisfaction
case_mixed = pd.DataFrame(
    [
        # Valid row + exact duplicate of it
        {"ticket_id": "T100", "opened_on": "2026-10-01", "channel": " Email ", "priority": "LOW", "handling_minutes": 0, "satisfaction": 5},
        {"ticket_id": "T100", "opened_on": "2026-10-01", "channel": " Email ", "priority": "LOW", "handling_minutes": 0, "satisfaction": 5},
        # Valid fractional duration + invalid fractional satisfaction
        {"ticket_id": "T101", "opened_on": "2026-10-31", "channel": "chat", "priority": "urgent", "handling_minutes": 12.5, "satisfaction": 4.5},
        # Conflicting ID pair (both must be rejected)
        {"ticket_id": "T200", "opened_on": "2026-10-10", "channel": "phone", "priority": "medium", "handling_minutes": 20, "satisfaction": 3},
        {"ticket_id": "T200", "opened_on": "2026-10-10", "channel": "web", "priority": "medium", "handling_minutes": 20, "satisfaction": 3},
        # Multiple rejection reasons: malformed ID + out-of-October date
        {"ticket_id": "BAD1", "opened_on": "2026-11-01", "channel": "web", "priority": "low", "handling_minutes": 15, "satisfaction": 2},
        # Missing date
        {"ticket_id": "T201", "opened_on": np.nan, "channel": "web", "priority": "low", "handling_minutes": 15, "satisfaction": 2},
        # Unparseable date
        {"ticket_id": "T202", "opened_on": "2026-10-32", "channel": "web", "priority": "low", "handling_minutes": 15, "satisfaction": 2},
    ]
)
c2, r2, f2, a2 = clean_tickets(case_mixed)
assert a2["raw_count"] == 8
assert a2["exact_duplicates_removed"] == 1
assert a2["rejected_count"] == 5  # 2 conflicting + 1 multi-reason + 1 missing date + 1 unparseable date
assert a2["retained_count"] == 2  # T100 and T101
assert a2["raw_count"] == a2["exact_duplicates_removed"] + a2["rejected_count"] + a2["retained_count"]
assert c2["handling_minutes"].tolist() == [0.0, 12.5]  # 0 and decimal are valid
assert c2["handling_imputed"].tolist() == [False, False]
assert pd.isna(c2.loc[c2["ticket_id"] == "T101", "satisfaction"].iloc[0])  # 4.5 rejected, not imputed
assert bool(c2.loc[c2["ticket_id"] == "T101", "satisfaction_unavailable"].iloc[0]) is True

# Verify multi-reason row recorded both reasons and was counted once
multi_row = r2.loc[r2["ticket_id"] == "BAD1", "rejection_reason"].iloc[0]
assert "malformed_id" in multi_row and "out_of_october_date" in multi_row

# Case 3: No valid retained durations must raise a clear ValueError
case_no_valid_dur = pd.DataFrame(
    [
        {"ticket_id": "T300", "opened_on": "2026-10-15", "channel": "email", "priority": "low", "handling_minutes": -5, "satisfaction": 3},
        {"ticket_id": "T301", "opened_on": "2026-10-16", "channel": "web", "priority": "high", "handling_minutes": "bad", "satisfaction": 4},
    ]
)
try:
    clean_tickets(case_no_valid_dur)
    raise AssertionError("Expected ValueError when no valid durations remain!")
except ValueError as err:
    print("Caught expected error for no valid durations:", err)

print("All edge-case tests passed!")

Caught expected error for no valid durations: No valid handling_minutes remain in retained rows; cannot compute imputation median.
All edge-case tests passed!


---

## 8. Export and report

Write required files to outputs/. Read the CSV files back, check row balance, and prepare REPORT.md.

In [10]:
# 8. Export deliverables to outputs/, read them back, and run acceptance checks
output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

clean_path = output_dir / "clean_tickets.csv"
rejected_path = output_dir / "rejected_tickets.csv"
features_path = output_dir / "ticket_features.csv"
audit_path = output_dir / "audit.json"

clean.to_csv(clean_path, index=False)
rejected.to_csv(rejected_path, index=False)
features.to_csv(features_path, index=False)
with open(audit_path, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2)

# Read all files back from disk to verify persistence and readability
clean_read = pd.read_csv(clean_path)
rejected_read = pd.read_csv(rejected_path)
features_read = pd.read_csv(features_path)
with open(audit_path, "r", encoding="utf-8") as f:
    audit_read = json.load(f)

# --- Acceptance checks ---
# 1. Row balance
assert audit_read["raw_count"] == (
    audit_read["exact_duplicates_removed"]
    + audit_read["rejected_count"]
    + audit_read["retained_count"]
), "Row balance failed!"
assert len(clean_read) == audit_read["retained_count"]
assert len(rejected_read) == audit_read["rejected_count"]

# 2. Retained IDs are nonempty, well formed, and unique
assert clean_read["ticket_id"].notna().all()
assert clean_read["ticket_id"].astype(str).str.fullmatch(r"T\d{3}").all()
assert clean_read["ticket_id"].is_unique

# 3. Dates are valid October 2026 dates
read_dates = pd.to_datetime(clean_read["opened_on"], errors="raise")
assert ((read_dates.dt.year == 2026) & (read_dates.dt.month == 10)).all()

# 4. Durations are finite and nonnegative
assert np.isfinite(clean_read["handling_minutes"]).all()
assert (clean_read["handling_minutes"] >= 0).all()

# 5. Categories are known values or 'unknown' only
assert set(clean_read["channel"]).issubset({"email", "phone", "web", "unknown"})
assert set(clean_read["priority"]).issubset({"low", "medium", "high", "unknown"})

# 6. Observed satisfaction ratings are integers between 1 and 5 (missing allowed by design)
observed_sat = clean_read["satisfaction"].dropna()
assert ((observed_sat >= 1) & (observed_sat <= 5) & (observed_sat % 1 == 0)).all()

# 7. Feature IDs align exactly with clean IDs; exactly one channel indicator active per row
indicator_cols = ["channel_email", "channel_phone", "channel_web", "channel_unknown"]
assert (features_read["ticket_id"].values == clean_read["ticket_id"].values).all()
assert (features_read[indicator_cols].sum(axis=1) == 1).all()
assert "satisfaction" not in features_read.columns

# 8. Standardized duration mean near 0, std near 1 (or all 0 if constant)
assert np.isclose(clean_read["handling_zscore"].mean(), 0.0, atol=1e-9)
if clean_read["handling_minutes"].nunique() > 1:
    assert np.isclose(clean_read["handling_zscore"].std(ddof=0), 1.0, atol=1e-9)
else:
    assert np.allclose(clean_read["handling_zscore"], 0.0)

print("All files exported, read back, and passed every acceptance check!")

All files exported, read back, and passed every acceptance check!
